# Tech Layoff Analysis — Phase 1: EDA + Cleaning

**Dataset:** swaptr/layoffs-2022 (Kaggle) — sourced from layoffs.fyi  
**Author:** Talib Hussain  
**Goal:** Understand the raw data shape, quality, and key distributions before cleaning.


## 1. Imports & Load
Import all libraries and load the raw CSV. We check shape and column names immediately to confirm the file loaded correctly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('../data/raw/layoffs_raw.csv')
print('✅ Loaded successfully')
print(f'Shape: {df.shape}')

## 2. First Look
Preview the first 10 rows to understand what the data looks like — column names, data types, and sample values.

In [ ]:
df.head(10)

## 3. Column Types & Nulls
Check data types for each column and count null values. The null % tells us which columns have real data quality issues that need to be addressed in cleaning.

In [ ]:
print('=== DTYPES ===')
print(df.dtypes)

print('\n=== NULL COUNTS ===')
print(df.isnull().sum())

print('\n=== NULL % ===')
print((df.isnull().sum() / len(df) * 100).round(1))

## 4. Shape & Unique Counts
Get a quick summary of how many unique companies, industries, countries, and stages exist. Also confirm the date range so we know what time period we are working with.

In [ ]:
print(f'Rows:              {df.shape[0]:,}')
print(f'Columns:           {df.shape[1]}')
print(f'\nUnique companies:  {df["company"].nunique():,}')
print(f'Unique industries: {df["industry"].nunique()}')
print(f'Unique countries:  {df["country"].nunique()}')
print(f'Unique stages:     {df["stage"].nunique()}')
print(f'\nDate range: {df["date"].min()} → {df["date"].max()}')

## 5. Value Counts — Categorical Columns
Inspect the distribution of industries, funding stages, and top countries. This helps catch label inconsistencies (e.g. 'Crypto' vs 'Cryptocurrency') that need to be normalized.

In [ ]:
print('=== INDUSTRIES ===')
print(df['industry'].value_counts())

print('\n=== STAGES ===')
print(df['stage'].value_counts())

print('\n=== TOP 15 COUNTRIES ===')
print(df['country'].value_counts().head(15))

## 6. Numeric Summary
Descriptive statistics for the three key numeric columns. Pay attention to the min/max values and the difference between count (non-null rows) and total rows — that gap is your null count.

In [ ]:
df[['total_laid_off', 'percentage_laid_off', 'funds_raised']].describe()

## 7. Top Companies by Layoffs
Aggregate total layoffs per company across all events. This gives us the all-time leaders in headcount cuts.

In [ ]:
df.groupby('company')['total_laid_off'] \
  .sum() \
  .sort_values(ascending=False) \
  .head(15)

## 8. Layoffs by Year
Parse the date column and extract year. Then aggregate total layoffs per year to see the macro trend — COVID wave vs 2022-23 correction vs AI restructuring.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

df.groupby('year')['total_laid_off'].sum().sort_index()

## 9. Chart — Layoffs by Year
Visualize the yearly totals as a bar chart. This is the headline chart of the project — it immediately shows which year was the most severe.

In [ ]:
yearly = df.groupby('year')['total_laid_off'].sum().dropna()

plt.figure(figsize=(10, 5))
plt.bar(yearly.index.astype(str), yearly.values, color='steelblue')
plt.title('Total Layoffs by Year', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Total Laid Off')
plt.tight_layout()
plt.show()

## 10. Spot Check — Companies That Fully Shut Down
Filter for rows where `percentage_laid_off == 1.0` — these companies eliminated their entire workforce. Sorted by funds raised to highlight the most capital-intensive shutdowns.

In [ ]:
df[df['percentage_laid_off'] == 1.0][
    ['company', 'industry', 'country', 'date', 'funds_raised']
].sort_values('funds_raised', ascending=False).head(20)

## 11. Chart — Companies That Shut Down (Top 20 by Funds Raised)
Horizontal bar chart of the top 20 shutdown companies ranked by how much money they raised before closing. The story: well-funded companies still failed.

In [ ]:
shutdowns = (
    df[df['percentage_laid_off'] == 1.0]
    .groupby('company', as_index=False)
    .agg(
        funds_raised=('funds_raised', 'max'),
        industry=('industry', 'first'),
        country=('country', 'first'),
        date=('date', 'max')
    )
    .dropna(subset=['funds_raised'])
    .sort_values('funds_raised', ascending=True)
    .tail(20)
)

fig, ax = plt.subplots(figsize=(11, 8))

bars = ax.barh(
    shutdowns['company'],
    shutdowns['funds_raised'],
    color='crimson',
    alpha=0.8
)

for bar, val in zip(bars, shutdowns['funds_raised']):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f'${val:,.0f}M', va='center', fontsize=8)

ax.set_title(
    'Top 20 Companies That Shut Down (100% Laid Off)\nRanked by Total Funding Raised',
    fontsize=13, fontweight='bold', pad=15
)
ax.set_xlabel('Funds Raised (Millions USD)', fontsize=11)
ax.set_ylabel('')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\nTotal companies with 100% layoffs: {len(df[df["percentage_laid_off"] == 1.0])}')